In [41]:
import networkx as nx

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline 
font = {'family' : 'DejaVu Sans',
        'weight' : 'bold',
        'size'   : 32}

plt.rc('font', **font)


In [42]:
import gzip

with gzip.open("protein.links.v12.0.txt.gz", "rt") as f:
    next(f)  # skip the header
    G0 = nx.read_weighted_edgelist(
        f,
        comments="#",
        nodetype=str
    )

df = pd.read_table("EssentialProteins_YeastMine_cerevisiae.csv", sep=",", header = None)
df.columns = ['counter','systematic name','organism','standard name','description']
essential_nodes = [n for n in G0.nodes() if n[5:] in list(df['systematic name'])]

threshold_score = 750
for edge in G0.edges: 
    weight = list(G0.get_edge_data(edge[0],edge[1]).values())
    if(weight[0] <= threshold_score):
        G0.remove_edge(edge[0],edge[1])

print('number of essential proteins:',len(essential_nodes))

number of essential proteins: 1304


In [43]:
import numpy as np
import networkx as nx
import scipy.sparse as sp


def rwr(G, seeds, restart=0.5, weight="weight", tol=1e-8, max_iter=1000):
    """Return {node: steady-state probability}. Higher = closer to the seeds."""
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    seeds = [s for s in seeds if s in idx]
    if not seeds:
        raise ValueError("None of the seeds are in the graph.")

    A = nx.to_scipy_sparse_array(G, nodelist=nodes, weight=weight, format="csr")  # adjacency matrix
    deg = np.asarray(A.sum(axis=0)).ravel()  # weighted degree of each node
    deg[deg == 0] = 1.0                      # avoid dividing by 0 for isolated nodes
    W = A @ sp.diags(1.0 / deg)              # column-normalised transition matrix

    p0 = np.zeros(len(nodes))
    p0[[idx[s] for s in seeds]] = 1.0 / len(seeds)   # restart vector: equal mass on every seed
    p = p0.copy()
    for _ in range(max_iter):
        p_next = (1 - restart) * (W @ p) + restart * p0
        if np.abs(p_next - p).sum() < tol:
            p = p_next
            break
        p = p_next
    return dict(zip(nodes, p))


def rank_candidates(G, seeds, **kw):
    """Rank non-seed nodes by RWR score."""
    scores = rwr(G, seeds, **kw)
    seedset = set(seeds)
    return sorted(((n, s) for n, s in scores.items() if n not in seedset),
                  key=lambda x: -x[1])


def leave_one_out(G, seeds, n_test=100, seed=0, **kw):
    """
    Validation: hide one seed, run RWR on the rest, and record the rank of the
    hidden seed among all non-seed nodes. Low median rank = the method works.
    """
    rng = np.random.default_rng(seed)
    seeds = [s for s in seeds if s in G]
    test = rng.choice(seeds, size=min(n_test, len(seeds)), replace=False)
    ranks = []
    for held in test:
        rest = [s for s in seeds if s != held]
        scores = rwr(G, rest, **kw)
        cand = {n: v for n, v in scores.items() if n not in set(rest)}
        order = sorted(cand, key=cand.get, reverse=True)
        ranks.append(order.index(held) + 1)
    return np.array(ranks), len(G) - len(seeds) + 1


G = G0
seeds = essential_nodes
top = rank_candidates(G, seeds)          # full ranking


In [44]:
N = nx.community.louvain_communities(G0, resolution=50, seed=555)
N = list(filter(lambda x : len(x) >= 3, N))
print(len(max(N)))

for i, community in enumerate(N, start=1):
    community_top = [
        (protein, score)
        for protein, score in top
        if protein in community
        ][:20]

print(f"\nCommunity {i}:")
for rank, (protein, score) in enumerate(community_top, start=1):
    print(rank, protein, score)

results = []

for community_num, community in enumerate(N, start=1):
    community_top = [
        (protein, score)
        for protein, score in top
        if protein in community
    ][:20]

for rank, (protein, score) in enumerate(community_top, start=1):
    results.append({
        "community": community_num,
        "rank": rank,
        "protein": protein,
        "RWR score": score
    })

pep5 = "4932.YMR231W"
pep5_neighbours = list(G0.neighbors(pep5))
adjacent_communities = set()

for i, community in enumerate(N, start=1):
    if any(node in community for node in pep5_neighbours):
        adjacent_communities.add(i)

print("PEP5 neighbours:", pep5_neighbours)
print("Adjacent communities:", sorted(adjacent_communities))

# Get top 5 RWR candidates for those communities
results = []

for community_num in sorted(adjacent_communities):
    community = N[community_num - 1]

    community_top = [
        (protein, score)
        for protein, score in top
        if protein in community
    ][:20]

    for rank, (protein, score) in enumerate(community_top, start=1):
        results.append([
            community_num,
            rank,
            protein,
            score
        ])

# results_df = pd.DataFrame(results)
# print(results_df)
        

6

Community 312:
1 4932.YOR394C-A 0.0
2 4932.YOR396W 0.0
3 4932.YOR396C-A 0.0
PEP5 neighbours: ['4932.YAL002W', '4932.YBR131W', '4932.YDL077C', '4932.YDL084W', '4932.YDR080W', '4932.YDR323C', '4932.YDR495C', '4932.YGL095C', '4932.YGL124C', '4932.YGL212W', '4932.YKR014C', '4932.YLR148W', '4932.YLR240W', '4932.YLR396C', '4932.YML001W', '4932.YMR197C', '4932.YOR036W', '4932.YOR089C', '4932.YOR106W', '4932.YNL304W', '4932.YPL045W']
Adjacent communities: [98, 139, 148, 226, 301]


In [45]:
pep5 = "4932.YMR231W"
pep5_neighbours = list(G0.neighbors(pep5))
adjacent_communities = set()

for i, community in enumerate(N, start=1):
    if any(node in community for node in pep5_neighbours):
        adjacent_communities.add(i)

print("PEP5 neighbours:", pep5_neighbours)
print("Adjacent communities:", sorted(adjacent_communities))

# Get top x RWR candidates for those communities

top_all = rank_candidates(G0, seeds)      # full ranking, no [:50]

results = []
for community_num in sorted(adjacent_communities):
    community = N[community_num - 1]
    community_top = [(p, s) for p, s in top_all if p in community][:20]
    print(f"Community {community_num}: size {len(community)}, returned {len(community_top)}")
    for rank, (protein, score) in enumerate(community_top, start=1):
        results.append([community_num, rank, protein, score])

results_df = pd.DataFrame(results, columns=["Community", "Rank", "Protein", "RWR Score"])

pd.DataFrame(top[:20], columns=["Protein", "RWR Score"]).rename_axis("Rank").reset_index().assign(Rank=lambda d: d.Rank + 1)

PEP5 neighbours: ['4932.YAL002W', '4932.YBR131W', '4932.YDL077C', '4932.YDL084W', '4932.YDR080W', '4932.YDR323C', '4932.YDR495C', '4932.YGL095C', '4932.YGL124C', '4932.YGL212W', '4932.YKR014C', '4932.YLR148W', '4932.YLR240W', '4932.YLR396C', '4932.YML001W', '4932.YMR197C', '4932.YOR036W', '4932.YOR089C', '4932.YOR106W', '4932.YNL304W', '4932.YPL045W']
Adjacent communities: [98, 139, 148, 226, 301]
Community 98: size 19, returned 18
Community 139: size 29, returned 20
Community 148: size 16, returned 12
Community 226: size 33, returned 14
Community 301: size 46, returned 20


,Rank,Protein,RWR Score
0,1,4932.YBR010W,0.000912
1,2,4932.YNL030W,0.000709
2,3,4932.YLL039C,0.000504
3,4,4932.YKL009W,0.000464
4,5,4932.YLR268W,0.000458
5,6,4932.YPL178W,0.000458
6,7,4932.YER029C,0.000452
7,8,4932.YPL129W,0.000428
8,9,4932.YMR125W,0.000416
9,10,4932.YDR225W,0.000416


In [50]:
from collections import Counter, defaultdict
import pandas as pd
import networkx as nx

pep5 = "4932.YMR231W"
representatives = results_df["Protein"].tolist()

raw_counts = Counter()                  # total paths through node (original metric)
weighted_counts = defaultdict(float)    # each representative contributes a total weight of 1
rep_coverage = Counter()                # number of representatives with >=1 path through node
n_reps_with_paths = 0

for protein in representatives:
    if protein not in G0 or pep5 not in G0:
        continue

    try:
        paths = list(nx.all_shortest_paths(G0, protein, pep5))
    except nx.NetworkXNoPath:
        continue

    n_reps_with_paths += 1
    n_paths = len(paths)
    nodes_seen = set()

    for path in paths:
        for node in path[1:-1]:          # exclude representative and PEP5
            raw_counts[node] += 1
            weighted_counts[node] += 1 / n_paths   # fraction of THIS rep's paths using node
            nodes_seen.add(node)

    for node in nodes_seen:
        rep_coverage[node] += 1

intermediary_df = pd.DataFrame({
    "Protein": list(raw_counts.keys()),
    "Raw Path Count": [raw_counts[n] for n in raw_counts],
    "Representatives Using": [rep_coverage[n] for n in raw_counts],
    "Relative Frequency": [weighted_counts[n] / n_reps_with_paths for n in raw_counts],
    "Fraction of Representatives": [rep_coverage[n] / n_reps_with_paths for n in raw_counts],
}).sort_values("Relative Frequency", ascending=False).reset_index(drop=True)

dist = nx.single_source_shortest_path_length(G0, pep5)   # {node: hops from PEP5}

intermediary_df = intermediary_df[
    intermediary_df["Protein"].map(lambda p: dist.get(p, float("inf")) >= 3)
].reset_index(drop=True)

display(intermediary_df)

,Protein,Raw Path Count,Representatives Using,Relative Frequency,Fraction of Representatives
0,4932.YOL110W,3,3,0.019345,0.035714
1,4932.YDR126W,12,2,0.019133,0.023810
2,4932.YDR431W,1,1,0.011905,0.011905
3,4932.YGL145W,31,1,0.011905,0.011905
4,4932.YDR264C,2,2,0.007440,0.023810
5,4932.YNL326C,1,1,0.001701,0.011905


In [47]:
def mean_comm_size(communities, min_size=3):
    """Mean size of communities with size >= min_size (i.e. drops size <= 4)."""
    sizes = [len(c) for c in communities if len(c) > 2]
    if not sizes:
        return None
    return sum(sizes) / len(sizes)

N = nx.community.louvain_communities(G0, resolution=50, seed=0)
print(mean_comm_size(N))

16.27936507936508
